In [ ]:
from dataclasses import dataclass
from typing import Literal

from dotenv import  load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import MessagesState, StateGraph,START,END
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from loguru import logger
from langchain.messages import HumanMessage,ToolMessage
from rich import print

import random

load_dotenv(override = True)


@tool(parse_docstring=True)
def get_weather(city:str)->str:
    """
    指定された都市の当日の天気を照会する

    Args:
        city:都市名
    """
    # 70%の確率で呼び出しが失敗する
    rand_int = random.randint(1,10)
    if rand_int < 8:
        logger.info("ネットワークが不安定です。再試行してください")
        raise ConnectionError("ネットワーク異常")
    return f"{city}は晴れ、微風です"

tools = [get_weather]

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)

model_with_tool = model.bind_tools(tools=tools)

#1. 最大リトライ回数を設定
@dataclass
class UserContext:
    max_attempts:int

# 状態として Messages をそのまま使用
#2. llm ノードを定義
def llm_node(state:MessagesState) -> MessagesState:
    messages = state["messages"]
    res = model_with_tool.invoke(messages)
    return {
        "messages":[res]
    }
#3. ルーターを定義。llm_node 実行後にツール呼び出しが必要かどうかを判定
def router(state:MessagesState) -> Literal["tool_node",END]:
    if state["messages"][-1].tool_calls:
        return "tool_node"
    return END

#4. ツール呼び出しラッパーを定義
# request: 固定パラメータ => runtime を呼び出せる
# execute: 実行 => ツール呼び出しを実行
def wrap_tool_call(request,execute):
    max_attempts = request.runtime.context.max_attempts
    tool_call_id = request.runtime.tool_call_id
    tool_msg = ""
    for i in range(max_attempts):
        try:
            # ツール呼び出しの前:
            logger.info("ツール呼び出し前")
            # 1回のツール呼び出しを表す
            tool_msg = execute(request)
            # ツール呼び出しの後
            logger.info("ツール呼び出し後")
            # 実行成功
            break
        except ConnectionError as e:
            logger.info("ツール呼び出しに失敗しました。現在の呼び出し回数:{}、呼び出し回数上限:{}、例外情報:{}",i+1,max_attempts,e)
    if not tool_msg:
        tool_msg = ToolMessage(
            tool_call_id = tool_call_id,
            content="呼び出し回数が上限に達しました。呼び出しに失敗しました"
        )
    print(f"tool_msg-->{tool_msg}")
    return tool_msg

#5. グラフを構築
builder = StateGraph(state_schema=MessagesState,context_schema=UserContext)

builder.add_node("llm_node",llm_node)
builder.add_node("tool_node",ToolNode(tools=tools,wrap_tool_call=wrap_tool_call))

builder.add_edge(START,"llm_node")
builder.add_conditional_edges("llm_node",router,path_map=["tool_node",END])
builder.add_edge("tool_node","llm_node")

graph = builder.compile()

from IPython.display import display
display(graph)

#6. 実行
res = graph.invoke({"messages":[HumanMessage(content="今日の東京の天気はどうですか？")]},context=UserContext(max_attempts=3))

print(res)
for msg in res["messages"]:
    msg.pretty_print()

